In [1]:
import numpy as np
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import pandas as pd
import glob 
import os
import pickle
from pathlib import Path
import time
import pandas as pd

import scipy.signal as signal
import scipy.fft as fft
import scipy.stats as stats

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import glob 
import pywt #PyWavelets package
import matplotlib.colors as colors
from matplotlib.colors import LogNorm



from data_process import select_df,read_and_process_data ,fac_decompose_dB

In [2]:
with open(Path("/Users/chengzhimao/Documents/Code/saturn2025/data/spice-location-ready/"+"cassini_enis_2004_2017_60s.pkl"), "rb") as f:
    cassini_location = pickle.load(f)

In [3]:
def merge_ICW_with_location(labels_df, cassini_location, tolerance='5min'):
    labels = labels_df.copy()
    loc = cassini_location.copy()

    labels['interval_start'] = pd.to_datetime(labels['interval_start']).dt.tz_localize(None)
    labels['interval_end'] = pd.to_datetime(labels['interval_end']).dt.tz_localize(None)
    loc['utc'] = pd.to_datetime(loc['utc']).dt.tz_localize(None)

    # Use interval midpoint as representative time
    labels['time'] = labels['interval_start'] + (
        labels['interval_end'] - labels['interval_start']
    ) / 2

    labels = labels.sort_values('time')
    loc = loc.sort_values('utc')

    # keep_cols = [
    #     'utc', 'x_ksm_Rs', 'y_ksm_Rs', 'z_ksm_Rs',
    #     'r_Rs', 'lat_deg', 'lon_deg'
    # ]
    # keep_cols = [c for c in keep_cols if c in loc.columns]

    merged = pd.merge_asof(
        labels,
        loc,
        left_on='time',
        right_on='utc',
        direction='nearest',
        tolerance=pd.Timedelta(tolerance)
    )

    # return merged.dropna(subset=['r_Rs']).reset_index(drop=True)
    return merged.reset_index(drop=True)
    

In [4]:
with open('data/check_wave/ICW_mirror_labels_2004_2017.pkl', 'rb') as f:
    labels_df = pickle.load(f)

In [5]:
ICW_loc_df = merge_ICW_with_location(labels_df, cassini_location)

In [6]:
ICW_loc_df

,interval_start,interval_end,is_ICW,is_mirror,label,file_name,B0_nT,fc_Hz_H2Oplus,icw_f_peak_Hz,icw_f_peak_over_fc,...,saturn_z_enis_km,saturn_x_enis_RE,saturn_y_enis_RE,saturn_z_enis_RE,sun_x_enis_km,sun_y_enis_km,sun_z_enis_km,sun_x_enis_RE,sun_y_enis_RE,sun_z_enis_RE
0,2004-07-01 00:10:00.500,2004-07-01 00:20:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,664.963930,0.567292,0.374376,0.659935,...,-1.089610e-12,6.912705e-14,943.626275,-4.323963e-15,1.200089e+09,-2.742972e+08,-5.605835e+08,4.762384e+06,-1.088510e+06,-2.224597e+06
1,2004-07-01 00:20:00.500,2004-07-01 00:30:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,777.296952,0.663126,0.374376,0.564563,...,-1.123629e-12,1.049396e-13,943.762162,-4.458964e-15,1.190720e+09,-3.124353e+08,-5.605880e+08,4.725204e+06,-1.239855e+06,-2.224615e+06
2,2004-07-01 00:30:00.500,2004-07-01 00:40:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,916.660681,0.782019,0.499168,0.638306,...,5.697729e-12,-4.844877e-14,943.898883,2.261063e-14,1.180142e+09,-3.502443e+08,-5.605924e+08,4.683228e+06,-1.389895e+06,-2.224632e+06
3,2004-07-01 00:40:00.500,2004-07-01 00:50:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,1091.602839,0.931266,0.499168,0.536010,...,-4.511631e-12,-3.582254e-15,944.036298,-1.790377e-14,1.168368e+09,-3.876862e+08,-5.605966e+08,4.636504e+06,-1.538478e+06,-2.224648e+06
4,2004-07-01 00:50:00.500,2004-07-01 01:00:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,1313.677714,1.120722,NaN,NaN,...,3.780981e-12,-1.431667e-13,944.174269,1.500429e-14,1.155410e+09,-4.247230e+08,-5.606005e+08,4.585083e+06,-1.685453e+06,-2.224664e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658981,2017-08-31 22:55:21.500,2017-08-31 23:05:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.626379,0.008212,0.006656,0.810426,...,-3.257067e-13,4.166228e-14,940.335944,-1.292521e-15,1.273147e+09,4.331465e+08,6.761990e+08,5.052303e+06,1.718881e+06,2.683401e+06
658982,2017-08-31 23:05:21.500,2017-08-31 23:15:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.538837,0.008138,0.006656,0.817864,...,2.243471e-12,-6.692418e-15,940.382560,8.902896e-15,1.286398e+09,3.920245e+08,6.761952e+08,5.104891e+06,1.555694e+06,2.683385e+06
658983,2017-08-31 23:15:21.500,2017-08-31 23:25:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.483587,0.008091,0.004992,0.616972,...,-1.586656e-12,5.224539e-14,940.433491,-6.296419e-15,1.298321e+09,3.505024e+08,6.761910e+08,5.152205e+06,1.390919e+06,2.683369e+06
658984,2017-08-31 23:25:21.500,2017-08-31 23:35:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.515213,0.008118,0.006656,0.819895,...,1.395176e-12,5.592884e-16,940.488683,5.536560e-15,1.308903e+09,3.086236e+08,6.761866e+08,5.194198e+06,1.224729e+06,2.683351e+06


# 3d

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pandas.api.types import is_bool_dtype, is_numeric_dtype


# ============================================================
# Constants
# ============================================================
R_SATURN_KM = 60268.0
R_ENCELADUS_KM = 252.1


# ============================================================
# Colorscales
# ============================================================
OCCURRENCE_COLORSCALE = [
    [0.00, "#e8f7ff"],
    [0.15, "#8be9fd"],
    [0.35, "#00b4d8"],
    [0.60, "#7b2cbf"],
    [0.82, "#f72585"],
    [1.00, "#ffb703"],
]

SATURN_COLORSCALE = [
    [0.00, "#5f3b1d"],
    [0.25, "#b87935"],
    [0.50, "#f1c27d"],
    [0.75, "#ffe0a6"],
    [1.00, "#fff2cc"],
]

ENCELADUS_COLORSCALE = [
    [0.00, "#6f8fa6"],
    [0.40, "#c7e9f1"],
    [1.00, "#ffffff"],
]

In [ ]:

# ============================================================
# Helper: event flag handling
# ============================================================
def make_event_indicator(series, event_values=None):
    """
    Convert an event column into 0/1 values.

    Examples
    --------
    Boolean flag:
        event_col = "is_ICW"
        event_values = None

    Label-based event:
        event_col = "label"
        event_values = ["ICW", "mixed"]
    """

    s = series.copy()

    if event_values is not None:
        if not isinstance(event_values, (list, tuple, set, np.ndarray, pd.Index)):
            event_values = [event_values]
        return s.isin(event_values).astype(float).to_numpy()

    if is_bool_dtype(s):
        return s.fillna(False).astype(float).to_numpy()

    if is_numeric_dtype(s):
        return (s.fillna(0) != 0).astype(float).to_numpy()

    return s.fillna("").astype(bool).astype(float).to_numpy()


# ============================================================
# Helper: axis limits
# ============================================================
def normalize_axis_limits(axis_limits, xyz):
    """
    Normalize axis limits into:
        [(xmin, xmax), (ymin, ymax), (zmin, zmax)]

    axis_limits can be:
        None
        scalar, e.g. 15
        list/tuple of 3 ranges, e.g. [(-10, 10), (-5, 5), (-2, 2)]
    """

    xyz = np.asarray(xyz, dtype=float)

    if axis_limits is None:
        limits = []
        for i in range(3):
            lo = np.nanmin(xyz[:, i])
            hi = np.nanmax(xyz[:, i])

            if not np.isfinite(lo) or not np.isfinite(hi):
                raise ValueError("Cannot infer axis limits from non-finite coordinates.")

            if lo == hi:
                lo -= 1.0
                hi += 1.0

            pad = 0.02 * (hi - lo)
            limits.append((lo - pad, hi + pad))

        return limits

    if np.isscalar(axis_limits):
        lim = float(axis_limits)
        return [(-lim, lim), (-lim, lim), (-lim, lim)]

    if len(axis_limits) != 3:
        raise ValueError("axis_limits must be None, a scalar, or 3 axis ranges.")

    return [tuple(axis_limits[0]), tuple(axis_limits[1]), tuple(axis_limits[2])]


In [21]:
ICW_loc_df

,interval_start,interval_end,is_ICW,is_mirror,label,file_name,B0_nT,fc_Hz_H2Oplus,icw_f_peak_Hz,icw_f_peak_over_fc,...,saturn_z_enis_km,saturn_x_enis_RE,saturn_y_enis_RE,saturn_z_enis_RE,sun_x_enis_km,sun_y_enis_km,sun_z_enis_km,sun_x_enis_RE,sun_y_enis_RE,sun_z_enis_RE
0,2004-07-01 00:10:00.500,2004-07-01 00:20:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,664.963930,0.567292,0.374376,0.659935,...,-1.089610e-12,6.912705e-14,943.626275,-4.323963e-15,1.200089e+09,-2.742972e+08,-5.605835e+08,4.762384e+06,-1.088510e+06,-2.224597e+06
1,2004-07-01 00:20:00.500,2004-07-01 00:30:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,777.296952,0.663126,0.374376,0.564563,...,-1.123629e-12,1.049396e-13,943.762162,-4.458964e-15,1.190720e+09,-3.124353e+08,-5.605880e+08,4.725204e+06,-1.239855e+06,-2.224615e+06
2,2004-07-01 00:30:00.500,2004-07-01 00:40:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,916.660681,0.782019,0.499168,0.638306,...,5.697729e-12,-4.844877e-14,943.898883,2.261063e-14,1.180142e+09,-3.502443e+08,-5.605924e+08,4.683228e+06,-1.389895e+06,-2.224632e+06
3,2004-07-01 00:40:00.500,2004-07-01 00:50:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,1091.602839,0.931266,0.499168,0.536010,...,-4.511631e-12,-3.582254e-15,944.036298,-1.790377e-14,1.168368e+09,-3.876862e+08,-5.605966e+08,4.636504e+06,-1.538478e+06,-2.224648e+06
4,2004-07-01 00:50:00.500,2004-07-01 01:00:00.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/04135_04240_00_...,1313.677714,1.120722,NaN,NaN,...,3.780981e-12,-1.431667e-13,944.174269,1.500429e-14,1.155410e+09,-4.247230e+08,-5.606005e+08,4.585083e+06,-1.685453e+06,-2.224664e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
658981,2017-08-31 22:55:21.500,2017-08-31 23:05:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.626379,0.008212,0.006656,0.810426,...,-3.257067e-13,4.166228e-14,940.335944,-1.292521e-15,1.273147e+09,4.331465e+08,6.761990e+08,5.052303e+06,1.718881e+06,2.683401e+06
658982,2017-08-31 23:05:21.500,2017-08-31 23:15:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.538837,0.008138,0.006656,0.817864,...,2.243471e-12,-6.692418e-15,940.382560,8.902896e-15,1.286398e+09,3.920245e+08,6.761952e+08,5.104891e+06,1.555694e+06,2.683385e+06
658983,2017-08-31 23:15:21.500,2017-08-31 23:25:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.483587,0.008091,0.004992,0.616972,...,-1.586656e-12,5.224539e-14,940.433491,-6.296419e-15,1.298321e+09,3.505024e+08,6.761910e+08,5.152205e+06,1.390919e+06,2.683369e+06
658984,2017-08-31 23:25:21.500,2017-08-31 23:35:21.500,False,False,uncertain,data/_cassini-mag-cal-sec-krtp/17242_17248_291...,9.515213,0.008118,0.006656,0.819895,...,1.395176e-12,5.592884e-16,940.488683,5.536560e-15,1.308903e+09,3.086236e+08,6.761866e+08,5.194198e+06,1.224729e+06,2.683351e+06


In [ ]:

# ============================================================
# 1. Statistics function
# ============================================================
def compute_3d_occurrence_statistics(
    df,
    coord_cols,
    event_col,
    event_values=None,
    coord_scale=1.0,
    n_bins=36,
    min_count=10,
    axis_limits=None,
    valid_col=None,
    event_name=None,
):
    """
    Compute 3D binned occurrence rate.

    Parameters
    ----------
    df : pandas.DataFrame
        Input merged dataframe.

    coord_cols : tuple/list of str
        Three coordinate columns to bin.
        Example:
            ("x_enis_km", "y_enis_km", "z_enis_km")

    event_col : str
        Column used to define the event.
        Example:
            "is_ICW", "is_mirror", or "label"

    event_values : None, str, or list
        If event_col is boolean, leave as None.
        If event_col is label-based, specify event labels.
        Example:
            event_col="label", event_values=["ICW", "mixed"]

    coord_scale : float
        Coordinates are divided by this value.
        Use R_SATURN_KM to convert km to Saturn radii.

    n_bins : int
        Number of bins per dimension.

    min_count : int
        Minimum number of samples required in a bin.

    axis_limits : None, scalar, or 3 ranges
        Binning region in scaled coordinate units.
        Example:
            15 means [-15, 15] in x, y, z.

    valid_col : str or None
        Optional validity column. If given, only rows with valid_col=True
        are used in the denominator.

    event_name : str or None
        Name used in metadata.

    Returns
    -------
    plot_df : pandas.DataFrame
        One row per surviving 3D bin.

    meta : dict
        Metadata describing the binning setup.
    """

    if len(coord_cols) != 3:
        raise ValueError("coord_cols must contain exactly three column names.")

    required_cols = list(coord_cols) + [event_col]
    if valid_col is not None:
        required_cols.append(valid_col)

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in input dataframe: {missing}")

    d = (
        df[required_cols]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=list(coord_cols) + [event_col])
        .copy()
    )

    if valid_col is not None:
        valid = make_event_indicator(d[valid_col]) > 0
        d = d.loc[valid].copy()

    if d.empty:
        raise ValueError("No valid rows available for binning.")

    xyz = d[list(coord_cols)].to_numpy(dtype=float) / coord_scale
    event = make_event_indicator(d[event_col], event_values=event_values)

    axis_ranges = normalize_axis_limits(axis_limits, xyz)

    keep = np.ones(len(d), dtype=bool)
    for i, (lo, hi) in enumerate(axis_ranges):
        keep &= (xyz[:, i] >= lo) & (xyz[:, i] <= hi)

    xyz = xyz[keep]
    event = event[keep]

    if len(xyz) == 0:
        raise ValueError("No rows remain inside the requested axis_limits.")

    bin_edges = [
        np.linspace(axis_ranges[0][0], axis_ranges[0][1], n_bins + 1),
        np.linspace(axis_ranges[1][0], axis_ranges[1][1], n_bins + 1),
        np.linspace(axis_ranges[2][0], axis_ranges[2][1], n_bins + 1),
    ]

    total_counts, edges = np.histogramdd(
        xyz,
        bins=bin_edges,
    )

    event_counts, _ = np.histogramdd(
        xyz,
        bins=bin_edges,
        weights=event,
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        occurrence_rate = event_counts / total_counts

    x_centers = 0.5 * (edges[0][:-1] + edges[0][1:])
    y_centers = 0.5 * (edges[1][:-1] + edges[1][1:])
    z_centers = 0.5 * (edges[2][:-1] + edges[2][1:])

    X, Y, Z = np.meshgrid(
        x_centers,
        y_centers,
        z_centers,
        indexing="ij",
    )

    mask = total_counts >= min_count

    plot_df = pd.DataFrame(
        {
            "x": X[mask],
            "y": Y[mask],
            "z": Z[mask],
            "n_total": total_counts[mask].astype(int),
            "n_event": event_counts[mask].astype(int),
            "occurrence_rate": occurrence_rate[mask],
        }
    )

    plot_df = plot_df[np.isfinite(plot_df["occurrence_rate"])].copy()

    if plot_df.empty:
        raise ValueError(
            "No bins survived min_count. "
            "Try lowering min_count or reducing n_bins."
        )

    meta = {
        "event_name": event_name if event_name is not None else event_col,
        "event_col": event_col,
        "event_values": event_values,
        "coord_cols": coord_cols,
        "coord_scale": coord_scale,
        "n_bins": n_bins,
        "min_count": min_count,
        "axis_ranges": axis_ranges,
        "n_input_rows": len(df),
        "n_rows_after_cleaning": len(d),
        "n_rows_inside_limits": len(xyz),
        "n_bins_survived": len(plot_df),
    }

    return plot_df, meta

In [ ]:

# ============================================================
# 2. Fixed body position function
# ============================================================
def compute_fixed_body_position(
    df,
    body_cols,
    coord_scale=1.0,
    method="median",
):
    """
    Compute a nearly fixed body position from dataframe columns.

    Example
    -------
    Saturn in Enceladus-centered ENIS, converted from km to R_S:

        saturn_pos = compute_fixed_body_position(
            df,
            body_cols=(
                "saturn_x_enis_km",
                "saturn_y_enis_km",
                "saturn_z_enis_km",
            ),
            coord_scale=R_SATURN_KM,
        )
    """

    if len(body_cols) != 3:
        raise ValueError("body_cols must contain exactly three columns.")

    missing = [c for c in body_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing body position columns: {missing}")

    arr = (
        df[list(body_cols)]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy(dtype=float)
    )

    if arr.size == 0:
        raise ValueError("No valid body position rows found.")

    arr = arr / coord_scale

    if method == "median":
        return np.nanmedian(arr, axis=0)

    if method == "mean":
        return np.nanmean(arr, axis=0)

    if method == "first":
        return arr[0]

    raise ValueError("method must be 'median', 'mean', or 'first'.")


# ============================================================
# 3. Geometry helpers
# ============================================================
def make_sphere(center, radius, n_theta=72, n_phi=36):
    center = np.asarray(center, dtype=float)

    theta = np.linspace(0, 2 * np.pi, n_theta)
    phi = np.linspace(0, np.pi, n_phi)

    theta, phi = np.meshgrid(theta, phi)

    x = center[0] + radius * np.sin(phi) * np.cos(theta)
    y = center[1] + radius * np.sin(phi) * np.sin(theta)
    z = center[2] + radius * np.cos(phi)

    return x, y, z


def make_shaded_sphere(
    center,
    radius,
    n_theta=72,
    n_phi=36,
    light_dir=np.array([-1.0, -0.6, 0.8]),
):
    x, y, z = make_sphere(
        center=center,
        radius=radius,
        n_theta=n_theta,
        n_phi=n_phi,
    )

    center = np.asarray(center, dtype=float)

    nx = (x - center[0]) / radius
    ny = (y - center[1]) / radius
    nz = (z - center[2]) / radius

    light_dir = np.asarray(light_dir, dtype=float)
    light_dir = light_dir / np.linalg.norm(light_dir)

    illumination = (
        nx * light_dir[0]
        + ny * light_dir[1]
        + nz * light_dir[2]
    )

    illumination = np.clip(illumination, -0.25, 1.0)
    illumination = (illumination - illumination.min()) / (
        illumination.max() - illumination.min()
    )

    return x, y, z, illumination


# ============================================================
# 4. Saturn + Enceladus plotting function
# ============================================================
def add_enis_reference_bodies(
    fig,
    saturn_pos,
    saturn_radius=1.0,
    enceladus_visual_radius=0.08, 
    show_enceladus=True,
    show_saturn=True,
    show_saturn_direction=True,
):
    """
    Add Enceladus at origin and Saturn at saturn_pos.

    Enceladus's is not to scale, but is visually larger than its true radius for clarity.
    
    Assumes coordinates are already in Saturn radii, R_S.
    Saturn physical radius is therefore 1.
    """

    saturn_pos = np.asarray(saturn_pos, dtype=float)

    if show_saturn:
        sat_x, sat_y, sat_z, sat_shade = make_shaded_sphere(
            center=saturn_pos,
            radius=saturn_radius,
            n_theta=72,
            n_phi=36,
        )

        fig.add_trace(
            go.Surface(
                x=sat_x,
                y=sat_y,
                z=sat_z,
                surfacecolor=sat_shade,
                colorscale=SATURN_COLORSCALE,
                cmin=0,
                cmax=1,
                showscale=False,
                opacity=0.92,
                lighting=dict(
                    ambient=0.35,
                    diffuse=0.85,
                    specular=0.25,
                    roughness=0.75,
                    fresnel=0.15,
                ),
                lightposition=dict(
                    x=-100,
                    y=-80,
                    z=120,
                ),
                hoverinfo="skip",
                name="Saturn",
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=[saturn_pos[0]],
                y=[saturn_pos[1]],
                z=[saturn_pos[2] + saturn_radius * 1.15],
                mode="text",
                text=["Saturn"],
                textposition="top center",
                textfont=dict(
                    size=14,
                    color="#8a4b08",
                ),
                hoverinfo="skip",
                showlegend=False,
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=[saturn_pos[0]],
                y=[saturn_pos[1]],
                z=[saturn_pos[2]],
                mode="markers",
                marker=dict(
                    size=3.5,
                    color="#3a2208",
                ),
                name="Saturn center",
                hovertext=[
                    "Saturn center"
                    + f"<br>x: {saturn_pos[0]:.3f} R<sub>S</sub>"
                    + f"<br>y: {saturn_pos[1]:.3f} R<sub>S</sub>"
                    + f"<br>z: {saturn_pos[2]:.3f} R<sub>S</sub>"
                ],
                hoverinfo="text",
            )
        )

    if show_enceladus:
        enc_x, enc_y, enc_z, enc_shade = make_shaded_sphere(
            center=np.array([0.0, 0.0, 0.0]),
            radius=enceladus_visual_radius,
            n_theta=48,
            n_phi=24,
        )

        fig.add_trace(
            go.Surface(
                x=enc_x,
                y=enc_y,
                z=enc_z,
                surfacecolor=enc_shade,
                colorscale=ENCELADUS_COLORSCALE,
                cmin=0,
                cmax=1,
                showscale=False,
                opacity=1.0,
                lighting=dict(
                    ambient=0.45,
                    diffuse=0.75,
                    specular=0.35,
                    roughness=0.5,
                    fresnel=0.2,
                ),
                lightposition=dict(
                    x=-100,
                    y=-80,
                    z=120,
                ),
                hoverinfo="skip",
                name="Enceladus",
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=[0],
                y=[0],
                z=[0],
                mode="text",
                text=["Enceladus"],
                textposition="top center",
                textfont=dict(
                    size=13,
                    color="black",
                ),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    if show_saturn_direction:
        fig.add_trace(
            go.Scatter3d(
                x=[0, saturn_pos[0]],
                y=[0, saturn_pos[1]],
                z=[0, saturn_pos[2]],
                mode="lines",
                line=dict(
                    color="rgba(255, 160, 40, 0.65)",
                    width=5,
                    dash="dash",
                ),
                name="Enceladus–Saturn direction",
                hoverinfo="skip",
            )
        )

    return fig

In [ ]:

# ============================================================
# 5. Plotting function
# ============================================================
def plot_3d_occurrence_rate(
    plot_df,
    meta=None,
    event_name="ICW",
    saturn_pos=None,
    axis_limits=15,
    unit_label="R<sub>S</sub>",
    title=None,
    occurrence_colorscale=OCCURRENCE_COLORSCALE,
    use_adaptive_color_scale=True,
    color_clip_percentile=98,
    marker_size_min=3.0,
    marker_size_max=11.0,
    add_reference_bodies=True,
    enceladus_visual_radius=0.08,
    output_html=None,
    show=True,
):
    """
    Plot 3D occurrence rate from output of compute_3d_occurrence_statistics().
    """

    required_cols = ["x", "y", "z", "n_total", "n_event", "occurrence_rate"]
    missing = [c for c in required_cols if c not in plot_df.columns]
    if missing:
        raise ValueError(f"plot_df is missing required columns: {missing}")

    p = plot_df.copy()

    if p.empty:
        raise ValueError("plot_df is empty.")

    # Marker size scaled by local sampling density
    p["marker_size"] = marker_size_min + (
        marker_size_max - marker_size_min
    ) * np.sqrt(p["n_total"] / p["n_total"].max())

    # Adaptive color limits
    cmin = 0.0

    if use_adaptive_color_scale:
        positive_rates = p.loc[p["occurrence_rate"] > 0, "occurrence_rate"]

        if len(positive_rates) > 0:
            cmax = np.nanpercentile(positive_rates, color_clip_percentile)
            cmax = max(cmax, p["occurrence_rate"].max() * 0.25)
            cmax = max(cmax, 0.01)
            cmax = min(cmax, 1.0)
        else:
            cmax = 1.0
    else:
        cmax = 1.0

    p["hover"] = (
        f"{event_name} occurrence rate: "
        + p["occurrence_rate"].map("{:.2%}".format)
        + f"<br>{event_name} samples: "
        + p["n_event"].astype(str)
        + "<br>Total samples: "
        + p["n_total"].astype(str)
        + "<br>x: "
        + p["x"].map("{:.2f}".format)
        + f" {unit_label}"
        + "<br>y: "
        + p["y"].map("{:.2f}".format)
        + f" {unit_label}"
        + "<br>z: "
        + p["z"].map("{:.2f}".format)
        + f" {unit_label}"
    )

    fig = go.Figure()

    fig.add_trace(
        go.Scatter3d(
            x=p["x"],
            y=p["y"],
            z=p["z"],
            mode="markers",
            marker=dict(
                size=p["marker_size"],
                color=p["occurrence_rate"],
                colorscale=occurrence_colorscale,
                cmin=cmin,
                cmax=cmax,
                opacity=0.82,
                line=dict(
                    width=0.2,
                    color="rgba(20,20,20,0.15)",
                ),
                colorbar=dict(
                    title=f"{event_name}<br>occurrence<br>rate",
                    tickformat=".1%",
                    len=0.72,
                    thickness=18,
                ),
            ),
            text=p["hover"],
            hoverinfo="text",
            name=f"{event_name} occurrence rate",
        )
    )

    if add_reference_bodies and saturn_pos is not None:
        add_enis_reference_bodies(
            fig,
            saturn_pos=saturn_pos,
            saturn_radius=1.0,
            enceladus_visual_radius=enceladus_visual_radius,
            show_enceladus=True,
            show_saturn=True,
            show_saturn_direction=True,
        )

    # Axis ranges
    xyz_for_limits = p[["x", "y", "z"]].to_numpy()

    if saturn_pos is not None:
        saturn_pos = np.asarray(saturn_pos, dtype=float)
        xyz_for_limits = np.vstack(
            [
                xyz_for_limits,
                saturn_pos + 1.0,
                saturn_pos - 1.0,
            ]
        )

    axis_ranges = normalize_axis_limits(axis_limits, xyz_for_limits)

    if title is None:
        title = (
            f"{event_name} Occurrence Rate in Enceladus-Centered ENIS Coordinates"
            f"<br><sup>Axes in Saturn radii, R<sub>S</sub>; "
            f"Enceladus at origin; Saturn position from dataframe</sup>"
        )

    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
        ),
        scene=dict(
            xaxis=dict(
                title=f"X ENIS [{unit_label}]",
                range=axis_ranges[0],
                backgroundcolor="rgba(248,250,252,1)",
                gridcolor="rgba(180,190,200,0.55)",
                zerolinecolor="rgba(80,80,80,0.8)",
                showspikes=False,
            ),
            yaxis=dict(
                title=f"Y ENIS [{unit_label}]",
                range=axis_ranges[1],
                backgroundcolor="rgba(248,250,252,1)",
                gridcolor="rgba(180,190,200,0.55)",
                zerolinecolor="rgba(80,80,80,0.8)",
                showspikes=False,
            ),
            zaxis=dict(
                title=f"Z ENIS [{unit_label}]",
                range=axis_ranges[2],
                backgroundcolor="rgba(248,250,252,1)",
                gridcolor="rgba(180,190,200,0.55)",
                zerolinecolor="rgba(80,80,80,0.8)",
                showspikes=False,
            ),
            aspectmode="cube",
            camera=dict(
                eye=dict(x=1.55, y=1.45, z=1.05),
            ),
        ),
        margin=dict(l=0, r=0, t=85, b=0),
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor="rgba(255,255,255,0.75)",
            bordercolor="rgba(180,180,180,0.5)",
            borderwidth=1,
        ),
        paper_bgcolor="white",
        font=dict(
            family="Arial",
            size=13,
        ),
    )

    if output_html is not None:
        fig.write_html(output_html)

    if show:
        fig.show()

    return fig

In [23]:
# Saturn position in Enceladus-centered ENIS frame, in R_S
saturn_pos = compute_fixed_body_position(
    df,
    body_cols=(
        "saturn_x_enis_km",
        "saturn_y_enis_km",
        "saturn_z_enis_km",
    ),
    coord_scale=R_SATURN_KM,
    method="median",
)

print("Saturn position [R_S]:", saturn_pos)
print("Saturn distance from Enceladus [R_S]:", np.linalg.norm(saturn_pos))

# 3D ICW occurrence statistics
icw_stats, icw_meta = compute_3d_occurrence_statistics(
    df,
    coord_cols=(
        "x_enis_km",
        "y_enis_km",
        "z_enis_km",
    ),
    event_col="is_ICW",
    event_values=None,
    coord_scale=R_SATURN_KM,
    n_bins=36,
    min_count=10,
    axis_limits=15,
    event_name="ICW",
)

# Plot
fig_icw = plot_3d_occurrence_rate(
    icw_stats,
    meta=icw_meta,
    event_name="ICW",
    saturn_pos=saturn_pos,
    axis_limits=15,
    unit_label="R<sub>S</sub>",
    output_html='output/waves/occurrence/3d/my_ICW.html',
    show=True,
)

Saturn position [R_S]: [1.40657352e-21 3.94973740e+00 9.08030585e-20]
Saturn distance from Enceladus [R_S]: 3.949737398993054


In [24]:
mirror_stats, mirror_meta = compute_3d_occurrence_statistics(
    df,
    coord_cols=(
        "x_enis_km",
        "y_enis_km",
        "z_enis_km",
    ),
    event_col="is_mirror",
    event_values=None,
    coord_scale=R_SATURN_KM,
    n_bins=36,
    min_count=10,
    axis_limits=15,
    event_name="Mirror-like",
)

fig_mirror = plot_3d_occurrence_rate(
    mirror_stats,
    meta=mirror_meta,
    event_name="Mirror-like",
    saturn_pos=saturn_pos,
    axis_limits=15,
    unit_label="R<sub>S</sub>",
    output_html='output/waves/occurrence/3d/my_mirror.html',
    show=True,
)